### Step 1: Environment setup

In [1]:
import os
import json
import time
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

print("PageIndex key loaded:", "✅" if PAGEINDEX_API_KEY else "❌ Missing!")
print("OpenAI key loaded:", "✅" if OPENAI_API_KEY else "❌ Missing!")

missing = [name for name in ("PAGEINDEX_API_KEY", "OPENAI_API_KEY") if not os.getenv(name)]
if missing:
    raise ValueError(f"Set {', '.join(missing)} in your .env file, then rerun this cell.")

PageIndex key loaded: ✅
OpenAI key loaded: ✅


In [3]:
from pageindex import PageIndexClient
from openai import OpenAI

# The current PageIndex SDK reads PAGEINDEX_API_KEY from the environment.
pi_client = PageIndexClient(index="cloud")
openai_client = OpenAI(api_key=OPENAI_API_KEY)

print("✅ PageIndex client initialized")
print("✅ OpenAI client initialized")
# Initialization does not verify credentials with the services.

✅ PageIndex client initialized
✅ OpenAI client initialized


### Step 2: Upload and index a PDF

Running the upload cell sends the selected PDF to PageIndex Cloud for processing
and returns a `doc_id` for later operations. Upload success does not mean indexing
has finished.

PageIndex uses a hierarchical document index, so this notebook does not create
fixed-size chunks or a vector database.

In [ ]:
# Locate the project whether the kernel starts in the root or notebook folder.
PROJECT_ROOT = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "2_RAG" / "data" / "pdf").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start the notebook from inside the LLM_Learning project.")

PDF_PATH = PROJECT_ROOT / "2_RAG" / "data" / "pdf" / "DS_takehome.pdf"

print(f"Uploading: {PDF_PATH}")
# 上传文件，同时发起建索引任务
# 云端接收后会排队、处理并建树；调用返回时，建树可能还没完成
result = pi_client.submit_document(str(PDF_PATH))
doc_id = result["doc_id"]

print("✅ Uploaded; indexing may still be in progress.")
print(f"Document ID: {doc_id}")
print("Save this ID — you will use it throughout the notebook.")


Uploading: /Users/bellahao/Desktop/LLM_Learning/2_RAG/data/pdf/DS_takehome.pdf
✅ Uploaded; indexing may still be in progress.
Document ID: pi-cmu3gzfui000m0cnscgn49yab
Save this ID — you will use it throughout the notebook.


### Wait for indexing to complete

Poll the existing `doc_id` every five seconds. If processing fails, stop with an
error; if it exceeds ten minutes, rerun the polling cell later using the same ID.
You do not need to upload the PDF again.


In [ ]:
POLL_INTERVAL_SECONDS = 5
TIMEOUT_SECONDS = 600

print("⏳ Building tree index...")
deadline = time.monotonic() + TIMEOUT_SECONDS #设定一个 600 秒后的截止点

while True:
    # 询处理进度，不触发建树。上一步已经开始indexing了
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"Status: {status}")

    if status == "completed":
        print("✅ Document processing completed!")
        break
    if status == "failed":
        raise RuntimeError(f"PageIndex processing failed for {doc_id}: {status_result}")
    if status not in {"queued", "processing"}:
        raise RuntimeError(f"Unexpected document status: {status_result}")

    remaining = deadline - time.monotonic()
    if remaining <= 0:
        raise TimeoutError(
            f"Processing is still pending after {TIMEOUT_SECONDS} seconds. "
            "Rerun this cell later with the same doc_id."
            #重新跑的话，就是仍然查询同一个 doc_id，
            #只是重新给本地等待计时。如果云端已经完成，第一次查询就会返回 completed
        )
    time.sleep(min(POLL_INTERVAL_SECONDS, remaining))

⏳ Building tree index...
Status: processing
Status: processing
Status: completed
✅ Document processing completed!


### Step 3: Inspect the tree structure

Fetch the tree with node summaries, count its top-level sections, and inspect the
first node as formatted JSON. Nested `nodes` represent subsections.
The full result remains available as `pageindex_tree` (and `tree`).


In [ ]:
# Fetch the full tree with node summaries.这里获取已经生成的树，并请求包含节点摘要
tree_result = pi_client.get_tree(doc_id, node_summary=True)
if tree_result.get("status") != "completed":
    raise RuntimeError(
        f"Tree is not ready: {tree_result.get('status')}. "
        "Run the polling cell before retrying."
    )

pageindex_tree = tree_result["result"]
# 给同一个树对象取另一个名字 tree，方便后面使用。没有复制数据，两个变量指向同一个对象
tree = pageindex_tree  # Keep the existing variable available for later cells.

# 打印顶层章节数量，不包括子章节
print(f"📊 Top-level sections: {len(pageindex_tree)}")

print("\n🌲 Raw tree (first node):")

# 把第一个顶层节点及其包含的子节点以易读的 JSON 格式打印出来：
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2, ensure_ascii=False))


📊 Top-level sections: 28

🌲 Raw tree (first node):
{
  "title": "Preface",
  "node_id": "0000",
  "page_index": 1,
  "summary": "The provided text comprises the title page, author details, and the complete table of contents for 'A Collection of Data Science Take-Home Challenges,' outlining numerous practical case studies and their corresponding solutions, along with promotional links for mentorship and feedback services.",
  "text": "Sold to\n\nQI.TIAN0123@GMAIL.COM\n\n5\n\n\"Best data science job interview resource\"\n\nDataScienceBootcamps.com\n\nA COLLECTION OF\nDATA SCIENCE\nTAKE-HOME CHALLENGES\n\n![img-0.jpeg](img-0.jpeg)\n\nGIULIO PALOMBO\n\n|  Intro | 4  |\n| --- | --- |\n|  Conversion Rate | 5  |\n|  Spanish Translation A/B Test | 7  |\n|  Employee Retention | 11  |\n|  Identifying Fraudulent Activities | 14  |\n|  Funnel Analysis | 18  |\n|  Pricing Test | 22  |\n|  Marketing Email Campaign | 25  |\n|  Song Challenge | 28  |\n\nClick here to check out our site if interested i

In [8]:
# Pretty-print the full tree.
def print_tree(nodes, indent=0):
    """Recursively print section titles, node IDs, and page numbers."""
    for node in nodes:
        prefix = "    " * indent + ("└─ " if indent > 0 else "")
        page = node.get("page_index", "?")
        node_id = node.get("node_id", "?")
        title = node.get("title", "Untitled")
        print(f"{prefix}[{node_id}] {title} (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

📚 Full Document Structure:

[0000] Preface (p.1)
[0001] Intro (p.5)
[0002] Conversion Rate (p.6)
[0003] Spanish Translation A/B Test (p.8)
[0004] Employee Retention (p.12)
[0005] Identifying Fraudulent Activities (p.15)
[0006] Funnel Analysis (p.19)
[0007] Pricing Test (p.23)
[0008] Marketing Email Campaign (p.26)
[0009] Song Challenge (p.29)
[0010] Clustering Grocery Items (p.31)
[0011] Credit Card Transactions (p.34)
[0012] User Referral Program (p.37)
[0013] Loan granting (p.40)
[0014] Json City Similarities (p.44)
[0015] Optimization of Employee Shuttle Stops (p.46)
[0016] Diversity in the Workplace (p.49)
[0017] URL Parsing Challenge (p.53)
[0018] Engagement Test (p.57)
[0019] On-Line Video Challenge (p.60)
[0020] Subscription Retention Rate (p.63)
[0021] Ads Analysis (p.66)
[0022] Solution: Conversion Rate (p.69)
[0023] Machine Learning (p.72)
[0024] Solution: Spanish Translation A/B Test (p.80)
[0025] Solution: Employee Retention (p.86)
[0026] Conclusions (p.93)
[0027] Solution:

In [9]:
# Count all nodes, including nested subsections.
def count_nodes(nodes):
    total = len(nodes)
    for node in nodes:
        if node.get("nodes"):
            total += count_nodes(node["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")
print("Each node represents a section of the document.")

🔢 Total nodes in tree: 29
Each node represents a section of the document.


### Step 4  LLM Tree Search — The Core of PageIndex

In [ ]:
# ── LLM Tree Search Function ─────────────────────────────────────────────────
# 这一步的逻辑是让LLM自己找出相关节点，并返回node编号
def llm_tree_search(query: str, tree: list, model: str = "gpt-4o") -> dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids.
    # 它现在只选章节，还没有读取选中章节的完整内容来回答问题。
    Returns: dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    """
    
    # Compress tree 压缩树 to save tokens — only send titles + short summaries
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]  # 截取 text 的前 150 个字符
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree) #返回一个tree node 的list
#=====================================================   
    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
Your task: identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.

Query: {query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY in this exact JSON format: #明确输出的内容
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""
#=====================================================
    response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        # 把整棵树压缩后，一次性放进模型的 context，让模型从中选相关节点，还没实现逐层检索
        response_format={"type": "json_object"}
    )
    
    return json.loads(response.choices[0].message.content)

In [11]:
# ── Test with a sample query ─────────────────────────────────────────────────
query = "What is the syllabus covered in Modern LLM finetuning?"

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))

🔍 Query: What is the syllabus covered in Modern LLM finetuning?

🧠 LLM Reasoning:
The query asks about the syllabus covered in Modern LLM finetuning. To find relevant sections, we should look for nodes that describe content related to learning, curriculum, machine learning, or fine-tuning models. Scanning the document tree, the closest match is node 0023 with the title 'Machine Learning'. Although not specifically about LLM finetuning, it discusses model building, which might cover elements relevant to the query.

🎯 Selected Node IDs: ['0023']


### 5: Full End-to-End RAG Pipeline

Tree Search → LLM picks relevant node_ids

Retrieve → Fetch the actual section content from those nodes

Generate → LLM writes a grounded answer with page citations

What makes this better than vector RAG:

Retrieved content has titles + page numbers (traceable)

LLM can cite exactly which section the answer comes from

No hallucination from irrelevant chunks

In [ ]:
# ── Helper: Find nodes by ID ─────────────────────────────────────────────────

def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    #检查每一个node，看一下他是不是属于 target_ids，如果是，就加入 found 列表；如果有子节点，就递归检查子节点
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [ ]:
# ── Generate answer from retrieved nodes ─────────────────────────────────────
# 这一步的目的是让LLM根据选出的节点内容来回答问题，并且在回答中引用章节标题和页码
def generate_answer(query: str, nodes: list, model: str = "gpt-4o") -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    if not nodes:
        return "⚠️ No relevant sections found in the document."
    
    # Build context string from retrieved nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n---\n\n".join(context_parts)
    
    prompt = f"""You are an expert document analyst.
Answer the question using ONLY the provided context.
For every claim you make, cite the section title and page number in parentheses.
Be concise and precise.

Question: {query}

Context:
{context}

Answer:"""
    
    response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

In [ ]:
# ── The complete Vectorless RAG function ─────────────────────────────────────

def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:
    
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")
    
    # Step 1: Tree Search
    # 这一步的逻辑是让LLM自己找出相关节点，并返回node编号
    search_result  = llm_tree_search(query, tree)
    node_ids       = search_result.get("node_list", [])
    
    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")
    
    # Step 2: Retrieve nodes
    # 这一步的逻辑是根据上一步选出的 node_ids，从树中找到对应的节点内容
    nodes = find_nodes_by_ids(tree, node_ids)
    
    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")
    
    # Step 3: Generate answer
    # 这一步的目的是让LLM根据选出的节点内容来回答问题，并且在回答中引用章节标题和页码
    answer = generate_answer(query, nodes)
    
    if verbose:
        print(f"\n📝 Answer:\n{answer}")
    
    return answer

In [16]:
# ── Run the full pipeline ────────────────────────────────────────────────────
answer = vectorless_rag(
    query="What's the key skill that this document emphasizes?",
    tree=pageindex_tree
)

🔍 Query: What's the key skill that this document emphasizes?

🧠 Reasoning: The query asks about the key skill emphasized in the document. To find this, we should look for sections discussing essential skills or skills required in the context of data science, since the docume...
🎯 Retrieved node IDs: ['0001', '0017']
📄 Sections found: ['Intro', 'URL Parsing Challenge']

📝 Answer:
The key skill that this document emphasizes is the ability to effectively parse and analyze data, particularly through URL parsing, and to derive meaningful insights for business impact. This includes skills like ensuring data reliability, data cleaning, extracting information, and simplifying solutions to focus on clear messaging and impacts (Section: 'Intro' | Page 5; Section: 'URL Parsing Challenge' | Page 53).
